# LLM 8일차 (1) — 싱글 에이전트 고객 응대 챗봇 : RAG 체인을 LangGraph 노드로 감싸기

수업 필기 `LangGraph_실습4` 를 정리 · 개선한 노트입니다. (교재 `AI Agent` **p.103~106** — 조건부 분기를 활용한 고객 응대 Chatbot Agent : 목표 · 과정 · 흐름)

```python
rag_chain = prompt | llm | StrOutputParser()                 # 지난주까지 만든 '직선' RAG 체인을

def rag_node(state: AgentState) -> dict:                     # 노드 하나로 포장
    question = state.messages[-1].content
    context = retriever.invoke(question)[0].page_content
    return {'messages': [AIMessage(rag_chain.invoke({'context': context, 'question': question}))]}

graph.add_node('rag_node', rag_node); graph.set_entry_point('rag_node'); graph.add_edge('rag_node', END)
```

- 교재 p.105 의 **조건부 분기판**(인사 · 일반 상식은 검색 없이 바로 답) 은 §7 에 이어서 만들었다
- 질문 재구성 · 문서 검색 · 답변을 나눠 맡는 **멀티 에이전트** 판은 [`langgraph_multi_agent_rag.ipynb`](./langgraph_multi_agent_rag.ipynb)
- 원본에서 고친 부분은 **📝 고친 부분** 으로 적었습니다. **애니메이션(GIF)** 은 [`images/`](./images/) 폴더에 있습니다.

> **출력 읽는 법** — 정리 노트북은 **API 키 없이** 만들었다.
>
> | 표시 | 뜻 |
> |---|---|
> | `# 📼` | **수업 시간에 실제로 실행한 결과**를 필기 원본에서 그대로 옮긴 출력 |
> | `# ⏳` | API 키가 필요해 **출력이 비어 있는 셀** — 직접 실행 |
> | `# 🧪` | OpenAI 대신 **로컬 임베딩**(`jhgan/ko-sroberta-multitask`) 또는 **가짜 LLM** 으로 실제로 돌린 셀 — 검색 순위 · 거리는 수업의 `text-embedding-3-small` 과 다를 수 있고, 답변 문장은 gpt-5-nano 와 다르다 |
> | 표시 없음 | 정리 노트북에서 실제로 실행한 셀 |

## 0. 설치 · 실행 방법

| 패키지 | 이 노트에서 쓰는 것 | 설치 |
|---|---|---|
| `langgraph` | `StateGraph` · `END` · `add_messages` | `pip install langgraph` |
| `langchain-openai` | `ChatOpenAI` · `OpenAIEmbeddings` | `pip install langchain-openai` |
| `langchain-community` | `CSVLoader` | `pip install langchain-community` |
| `langchain-chroma` | `Chroma` 벡터 저장소 | `pip install langchain-chroma` |
| `langchain-huggingface` | (🧪 키 없이 실행할 때) 로컬 임베딩 | `pip install langchain-huggingface sentence-transformers` |

1. 노트북과 같은 폴더에 **`faq_manual.csv`**(이 폴더에 있음)와 `.env`(`OPENAI_API_KEY=…`)를 둔다 (**`.env` 는 올리지 않는다** — 저장소 `.gitignore` 에 포함)
2. 키가 있으면 `# ⏳` 셀을 실행하고 `# 🧪` 셀은 건너뛴다. 키가 없으면 그 반대로 — 끝까지 실행된다
3. 정리 환경 : `langgraph 1.2.21` · `langchain-core 1.8.0` · `langchain-community 0.4.2` · `langchain-chroma 1.1.0` · `langchain-openai 1.7.4`

> `langchain-community` 를 import 하면 '지원 종료(sunset) 예정' 경고가 나온다(필기 출력에도 있음). 오늘 코드는 그대로 동작한다.

In [1]:
# ① 설치 (처음 한 번)
# !pip install -q langgraph langchain-openai langchain-community langchain-chroma python-dotenv

In [2]:
import os, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
from typing import Annotated
from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_community.document_loaders import CSVLoader
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import add_messages, StateGraph, END

In [3]:
# .env 파일 → 환경 변수 (키 값은 절대 print 하지 않는다)
loaded = load_dotenv(find_dotenv(usecwd=True), override=True)
print('.env 읽음 :', loaded, '| OPENAI_API_KEY 있음 :', bool(os.getenv('OPENAI_API_KEY')))

.env 읽음 : False | OPENAI_API_KEY 있음 : False


## 1. 데이터 — FAQ 10개 (`faq_manual.csv`)

| 열 | 예 |
|---|---|
| `faq_id` | FAQ-001 ~ FAQ-010 |
| `category` | 회원/계정 · 결제/환불 · 수강/수료 · 기술지원 · 수강/자료 · 학습지원 |
| `question` · `answer` | 가입 · 환불 · 수료증 · 결제 수단 · 동영상 재생 · 자료 다운로드 · 수강 기간 · 질문 게시판 · 입문 강의 · 모바일 앱 |

In [4]:
# CSV → Document (한 행 = 문서 1개) — 본문에는 질문 · 답, 메타데이터에는 번호 · 분류
loader = CSVLoader(
    file_path='faq_manual.csv',
    encoding='utf-8',
    content_columns=['question', 'answer'],          # page_content 에 넣을 열
    metadata_columns=['faq_id', 'category'],         # 📝 metadata 에 넣을 열 (필기에는 없음)
)
docs = loader.load()
print(f'로드된 문서 수 : {len(docs)}개')
print(docs[1].page_content)
print(docs[1].metadata)

로드된 문서 수 : 10개
question: 수강 신청한 강의를 환불하고 싶어요. 환불 정책은 어떻게 되나요?
answer: 네, 환불 가능합니다. 결제일로부터 7일 이내, 그리고 강의 수강률이 5% 미만일 경우 전액 환불해 드리고 있습니다. 고객센터를 통해 문의하시면 신속하게 처리해 드리겠습니다.
{'source': 'faq_manual.csv', 'row': 1, 'faq_id': 'FAQ-002', 'category': '결제/환불'}


> **📝 보충** 필기처럼 `content_columns` 만 주면 메타데이터는 `{'source', 'row'}` 뿐이다(필기 출력). `faq_id` · `category` 가 본문에도 메타데이터에도 **아예 들어가지 않는다**. `metadata_columns` 를 주면 '어느 FAQ 를 찾았나' 를 번호로 확인할 수 있고, Day 050 처럼 `category` 로 필터도 걸 수 있다. 교재 p.104 의 '텍스트 분할' 은 하지 않았다 — 한 행이 2~3문장이라 나눌 필요가 없다.

## 2. 임베딩 · 벡터 저장소 · 검색기

In [ ]:
# ⏳ API 키가 필요한 셀 — 정리 환경에서는 실행하지 않음 (직접 실행해 보기)

from langchain_openai import OpenAIEmbeddings
model = OpenAIEmbeddings(model='text-embedding-3-small')    # 1536차원 · 입력 최대 8,191 토큰

In [6]:
# 🧪 OpenAI 임베딩 대신 로컬 한국어 임베딩(jhgan/ko-sroberta-multitask)으로 실제 실행 — 순위 · 거리는 text-embedding-3-small 과 다를 수 있다

from langchain_huggingface import HuggingFaceEmbeddings
model = HuggingFaceEmbeddings(model_name='jhgan/ko-sroberta-multitask',      # 처음 한 번 약 440MB 내려받음
                              encode_kwargs={'normalize_embeddings': True})
print('벡터 길이 :', len(model.embed_query('테스트')))

벡터 길이 : 768


In [7]:
# 필기 : 폴더가 있으면 '불러오기', 없으면 '새로 만들기'
chroma_path = 'chroma_db_faq_local'          # 📝 데이터 · 임베딩 모델마다 다른 폴더 이름

if os.path.exists(chroma_path) and os.listdir(chroma_path):
    vectorstore = Chroma(persist_directory=chroma_path, embedding_function=model)
    print('벡터 저장소 불러오기')
else:
    vectorstore = Chroma.from_documents(documents=docs, embedding=model, persist_directory=chroma_path)
    print('벡터 저장소 새로 생성')
print('저장된 문서 수 :', vectorstore._collection.count())

벡터 저장소 새로 생성
저장된 문서 수 : 10


In [8]:
# 이 확인(if) 없이 from_documents 를 한 번 더 부르면?
again = Chroma.from_documents(documents=docs, embedding=model, persist_directory=chroma_path)
print('같은 폴더에 다시 from_documents → 문서 수 :', again._collection.count())

같은 폴더에 다시 from_documents → 문서 수 : 20


- `from_documents` 는 **덧붙이기**라 같은 FAQ 가 두 벌(20개)이 된다 → 필기의 '있으면 불러오기' 확인이 필요한 이유
- 📝 반대로 그 확인 때문에 **CSV 를 고쳐도 옛 벡터를 계속 쓴다**. 필기 출력 '백터 저장소 불러오기 성공' 은 수업 중 앞서 만든 DB 를 읽은 것이다. 데이터 · 임베딩 모델을 바꾸면 **폴더 이름을 바꾸거나 폴더를 지우고** 다시 만들자(`chroma_db_rag` 처럼 일반적인 이름은 다른 실습 DB 와 섞이기 쉽다). 같은 실행 안에서 지우면 readonly 오류가 나므로(Day 049) 새 이름이 안전하다
- 오타 `백터` → 벡터

In [9]:
# 중복이 생긴 저장소를 비우고 다시 10개로 (다음 셀부터 쓰는 저장소)
vectorstore = Chroma.from_documents(documents=docs, embedding=model, collection_name='faq_clean')
retriever = vectorstore.as_retriever(search_kwargs={'k': 1})       # 가장 가까운 FAQ 1개
print('문서 수 :', vectorstore._collection.count())

문서 수 : 10


In [10]:
# 🧪 OpenAI 임베딩 대신 로컬 한국어 임베딩(jhgan/ko-sroberta-multitask)으로 실제 실행 — 순위 · 거리는 text-embedding-3-small 과 다를 수 있다

# 검색 확인 — k=1 검색기가 무엇을 고르는지 (거리 : 작을수록 가깝다)
tests = ['환불은 어떻게?', '수료증 받을 수 있어?', '강의 영상이 안 나와요',
         '그거 전액 다 받으려면 조건이 뭐야?', '안녕하세요', '오늘 날씨 어때?']
for q in tests:
    (doc, dist), = vectorstore.similarity_search_with_score(q, k=1)
    print(f"{q:<22} → {doc.metadata['faq_id']} ({doc.metadata['category']}) 거리 {dist:.3f} | {doc.page_content.splitlines()[0][10:40]}")

환불은 어떻게?               → FAQ-002 (결제/환불) 거리 0.814 | 수강 신청한 강의를 환불하고 싶어요. 환불 정책은 어떻
수료증 받을 수 있어?           → FAQ-003 (수강/수료) 거리 0.684 | 강의를 다 들으면 수료증을 받을 수 있나요?
강의 영상이 안 나와요           → FAQ-005 (기술지원) 거리 0.957 | 강의 동영상이 자꾸 끊기거나 재생되지 않아요.


그거 전액 다 받으려면 조건이 뭐야?   → FAQ-003 (수강/수료) 거리 1.048 | 강의를 다 들으면 수료증을 받을 수 있나요?
안녕하세요                  → FAQ-008 (학습지원) 거리 1.385 | 코딩 실습 중에 모르는 것이 생기면 어떻게 하죠?


오늘 날씨 어때?              → FAQ-010 (기술지원) 거리 1.636 | 모바일 앱으로도 강의를 들을 수 있나요?


- 앞의 세 질문은 맞는 FAQ(환불 · 수료증 · 동영상)를 찾았다
- **`k=1` 검색기는 관계없는 질문에도 반드시 1개를 돌려준다** — '안녕하세요' → 질문 게시판(FAQ-008), '오늘 날씨' → 모바일 앱(FAQ-010). 이 글이 그대로 '컨텍스트' 로 LLM 에 들어간다(프롬프트 규칙 3번이 막아 주길 기대할 뿐) → 교재 p.105 가 **검색 전에 분류 노드**를 두는 이유(§7)
- '**그거** 전액 다 받으려면' 은 앞 대화를 모르면 무엇인지 알 수 없어 수료증(FAQ-003)을 찾았다 → 멀티 에이전트 노트의 질문 재구성

## 3. RAG 체인 (필기)

In [11]:
system_prompt = '''
당신은 친절한 온라인 교육 플랫폼 직원입니다.
제공된 [컨텍스트]를 바탕으로 고객의 질문에 정중하게 답변해주세요.

[답변 규칙]
1. [컨텍스트]의 내용을 바탕으로 자연스럽게 답변을 재구성하거나 그대로 활용하세요.
2. 답변 시 '답변:'이라는 표현을 사용하지 마세요.
3. [컨텍스트]에 질문에 대한 정보가 없다면, '죄송하지만 문의하신 내용에 대해서는 아는 바가 없습니다.'라고 답변하세요.

[컨텍스트]
{context}
'''
prompt = ChatPromptTemplate.from_messages([('system', system_prompt), ('human', '{question}')])
print('입력 변수 :', prompt.input_variables)

입력 변수 : ['context', 'question']


In [ ]:
# ⏳ API 키가 필요한 셀 — 정리 환경에서는 실행하지 않음 (직접 실행해 보기)

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model='gpt-5-nano')

In [13]:
# 🧪 LLM 대신 '정해 둔 규칙으로 답하는' 가짜 LLM — 그래프에 무엇이 들어가고 어느 길로 가는지 확인용

from langchain_core.runnables import RunnableLambda

def _fake_llm(prompt_value):
    msgs = prompt_value.to_messages()
    system, question = msgs[0].content, msgs[-1].content
    context = system.split('[컨텍스트]')[-1].strip()
    answer = context.split('answer: ')[-1].split('. ')[0] if 'answer: ' in context else '(컨텍스트 없음)'
    return AIMessage(f'(모의 답변 · 컨텍스트의 첫 문장) {answer}.')

llm = RunnableLambda(_fake_llm)

In [14]:
rag_chain = prompt | llm | StrOutputParser()            # LCEL : 프롬프트 → LLM → 글자
print(rag_chain.invoke({'context': docs[1].page_content, 'question': '환불은 어떻게?'}))

(모의 답변 · 컨텍스트의 첫 문장) 네, 환불 가능합니다.


## 4. 상태 · 노드 · 그래프 (교재 p.104 의 3~6단계 · 필기)

In [15]:
class AgentState(BaseModel):
    messages: Annotated[list[BaseMessage], add_messages] = []     # 상태는 대화 기록 하나

def rag_node(state: AgentState) -> dict:
    question = state.messages[-1].content                        # 최신 사용자 질문
    retrieved_docs = retriever.invoke(question)                  # [Document] 1개
    context_text = retrieved_docs[0].page_content if retrieved_docs else ''   # 📝 빈 결과면 IndexError 방지
    print(f"  [rag_node] {question} → {retrieved_docs[0].metadata.get('faq_id') if retrieved_docs else '없음'}")
    response = rag_chain.invoke({'context': context_text, 'question': question})
    return {'messages': [AIMessage(content=response)]}

graph = StateGraph(AgentState)
graph.add_node('rag_node', rag_node)
graph.set_entry_point('rag_node')
graph.add_edge('rag_node', END)
agent = graph.compile()
print(agent.get_graph().draw_mermaid().split('graph TD;')[1].split('classDef')[0])


	__start__([<p>__start__</p>]):::first
	rag_node(rag_node)
	__end__([<p>__end__</p>]):::last
	__start__ --> rag_node;
	rag_node --> __end__;
	


- 노드 하나 · 직선(`START → rag_node → END`) — **'싱글 에이전트'** 다. 체인 하나를 노드로 감쌌을 뿐이라 동작은 `rag_chain` 과 같지만, 이제 이 노드에 다른 노드 · 조건부 엣지를 붙일 수 있다(교재 p.103 '부품으로 재사용')

## 5. 실행 — 필기의 `while` + `input()` 챗봇

In [ ]:
# ⏳ API 키가 필요한 셀 — 정리 환경에서는 실행하지 않음 (직접 실행해 보기)

# 필기 그대로 — 질문을 직접 입력하는 대화 루프 ('exit' 로 종료). 노트북 자동 실행에서는 input() 이 멈추므로 실행하지 않았다
print("안녕하세요! 코드크래프터스 아카데미 지원 챗봇입니다. 무엇이든 물어보세요! (종료하려면 'exit' 입력)")
while True:
    try:
        user_input = input('질문: ')
        if user_input.strip().lower() == 'exit':                  # 📝 앞뒤 공백도 허용
            print('챗봇을 종료합니다. 이용해 주셔서 감사합니다.')
            break
        final_state = agent.invoke(AgentState(messages=[HumanMessage(content=user_input)]))
        print(f"[Chatbot Agent 응답] \n{final_state['messages'][-1].content}")
        print('-' * 80)
    except Exception as e:
        print(f'오류가 발생했습니다: {e}')

In [17]:
# 🧪 LLM 대신 '정해 둔 규칙으로 답하는' 가짜 LLM — 그래프에 무엇이 들어가고 어느 길로 가는지 확인용

# 같은 루프를 '입력 목록' 으로 흉내 — input() 대신 정해 둔 질문을 차례로 넣는다
scripted = iter(['환불은 어떻게?', '그거 전액 다 받으려면 조건이 뭐야?', 'exit'])
while True:
    user_input = next(scripted)
    print(f'질문: {user_input}')
    if user_input.strip().lower() == 'exit':
        print('챗봇을 종료합니다.'); break
    final_state = agent.invoke(AgentState(messages=[HumanMessage(content=user_input)]))
    print(f"  messages 수 : {len(final_state['messages'])} | 응답 : {final_state['messages'][-1].content}\n")

질문: 환불은 어떻게?
  [rag_node] 환불은 어떻게? → FAQ-002
  messages 수 : 2 | 응답 : (모의 답변 · 컨텍스트의 첫 문장) 네, 환불 가능합니다.

질문: 그거 전액 다 받으려면 조건이 뭐야?


  [rag_node] 그거 전액 다 받으려면 조건이 뭐야? → FAQ-003
  messages 수 : 2 | 응답 : (모의 답변 · 컨텍스트의 첫 문장) 물론입니다.

질문: exit
챗봇을 종료합니다.


- 매 질문마다 `AgentState(messages=[지금 질문])` 으로 **새로 시작**한다 → `messages` 는 늘 [질문, 답] 2개. 두 번째 질문의 '그거' 가 환불이라는 것을 그래프가 알 길이 없어 **수료증 FAQ** 로 답했다
- 대화를 이어 가려면 ① 앞 결과의 `messages` 에 새 질문을 붙여 넣고(멀티 에이전트 노트 §5) ② 그 기록으로 **질문을 다시 쓰는** 노드가 필요하다 — 다음 노트
- `agent.invoke(AgentState(...))` 처럼 BaseModel 객체를 넣어도 되고 `{'messages': [...]}` dict 를 넣어도 된다. 결과는 언제나 **dict**

## 6. 📝 교재 p.105 흐름 — 검색이 필요할 때만 검색 (조건부 분기판)

교재의 흐름도는 `classifier_node`(RAG / NO_RAG 판별) → `route_by_rag_need` → `retriever_node`(ChromaDB 매뉴얼 검색) → `answer_node` 다. 필기 `LangGraph_실습4` 는 분기 없이 노드 하나라서, Day 052 의 검색 분기 에이전트와 같은 방법으로 교재 흐름을 만들었다.

In [18]:
# 🧪 LLM 대신 '정해 둔 규칙으로 답하는' 가짜 LLM — 그래프에 무엇이 들어가고 어느 길로 가는지 확인용

class RagState(BaseModel):
    messages: Annotated[list[BaseMessage], add_messages] = []
    needs_rag: bool = False
    context: str = ''

GREETING = ('안녕', '고마워', '감사', '반가')

def classifier_node(state: RagState) -> dict:
    # 진짜 실습에서는 LLM 에게 'RAG' / 'NO_RAG' 단답을 받는다 (Day 052 의 SEARCH / NO_SEARCH 와 같은 방법)
    q = state.messages[-1].content
    return {'needs_rag': not any(w in q for w in GREETING)}

def retriever_node(state: RagState) -> dict:
    doc = retriever.invoke(state.messages[-1].content)[0]
    return {'context': doc.page_content}

def answer_node(state: RagState) -> dict:
    q = state.messages[-1].content
    if state.needs_rag and state.context:
        return {'messages': [AIMessage(rag_chain.invoke({'context': state.context, 'question': q}))]}
    return {'messages': [AIMessage('(모의 답변 · 검색 없이) 안녕하세요! 코드크래프터스 아카데미 지원 챗봇입니다.')]}

def route_by_rag_need(state: RagState) -> str:
    return 'retriever_node' if state.needs_rag else 'answer_node'

workflow = StateGraph(RagState)
workflow.add_node('classifier_node', classifier_node)
workflow.add_node('retriever_node', retriever_node)
workflow.add_node('answer_node', answer_node)
workflow.set_entry_point('classifier_node')
workflow.add_conditional_edges('classifier_node', route_by_rag_need,
                               {'retriever_node': 'retriever_node', 'answer_node': 'answer_node'})
workflow.add_edge('retriever_node', 'answer_node')
workflow.add_edge('answer_node', END)                  # 📝 교재 p.105 의 work.add_edge → workflow.add_edge
rag_agent = workflow.compile()

for q in ['안녕하세요', '환불은 어떻게?']:
    path = []
    for u in rag_agent.stream({'messages': [HumanMessage(q)]}, stream_mode='updates'):
        path.append(next(iter(u)))
    out = rag_agent.invoke({'messages': [HumanMessage(q)]})
    print(f"{q:<10} 경로 : {' → '.join(path)}\n           답 : {out['messages'][-1].content}")

안녕하세요      경로 : classifier_node → answer_node
           답 : (모의 답변 · 검색 없이) 안녕하세요! 코드크래프터스 아카데미 지원 챗봇입니다.


환불은 어떻게?   경로 : classifier_node → retriever_node → answer_node
           답 : (모의 답변 · 컨텍스트의 첫 문장) 네, 환불 가능합니다.


- '안녕하세요' 는 검색을 건너뛰어 엉뚱한 FAQ(질문 게시판)가 컨텍스트로 들어가지 않는다
- 📝 교재 p.105 그림의 `work.add_edge("answer_node", END)` 는 Day 052 (p.101) 과 같은 변수 이름 오타 → `workflow.add_edge`

## 정리

| 할 일 | 코드 |
|---|---|
| CSV → 문서 | `CSVLoader(file, content_columns=[...], metadata_columns=[...])` — 한 행 = 문서 1개 |
| 저장소 | 폴더가 있으면 불러오기 (`from_documents` 를 다시 부르면 **중복**) · 데이터를 바꾸면 **새 폴더** |
| 체인 → 노드 | 노드 안에서 `retriever.invoke` → `rag_chain.invoke` → `{'messages': [AIMessage]}` |
| 싱글 에이전트 | `START → rag_node → END` — 질문마다 새 상태라 **앞 대화를 모른다** |
| 교재 p.105 | `classifier → (needs_rag) → retriever → answer` — 인사에는 검색하지 않음 |